In [25]:
from openai import AzureOpenAI, OpenAI
import tiktoken

# from markitdown import MarkItDown
# from langchain_text_splitters import MarkdownTextSplitter, RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import PyPDFLoader

import psycopg2
# from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector

# import numpy as np
import pdfplumber
# from sentence_transformers import SentenceTransformer

from dotenv import load_dotenv
import os
# import re

from utils.chunking import *
from utils.embedding import *

load_dotenv("project.env")

True

In [2]:
import gc
import torch

# delete big GPU objects if they exist
for var_name in ["wrapper", "embeddings", "model"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print(torch.cuda.memory_summary() if torch.cuda.is_available() else "CUDA not available")

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |

In [5]:
def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text


In [3]:
path = "./data/"
fname = "euaiact.pdf"
pdf_path = os.path.join(path, fname)

In [6]:
# read the pdf and extract text
pdf_text = extract_text_from_pdf(pdf_path)

In [7]:
res = chunk_semantically_spacy(pdf_text)

In [6]:
# count tokens in the first chunk
tokenizer = tiktoken.get_encoding("cl100k_base")
print(len(res), len(tokenizer.encode(res[3])))

503 175


In [49]:
# These are small'ish models, ok to run locally.
# Bigger models should be deployed to Azure and used via . See .env for deployment names det's.
names = [
    'BAAI/bge-base-en-v1.5', # 1024 
    'BAAI/bge-large-en-v1.5', # 1024
    'sentence-transformers/all-MiniLM-L6-v2' # 384 THIS IS mini 
]

In [44]:
wrapper = SentenceTransformerWrapper(names[1])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
embeddings = wrapper.embed_documents(res)

In [52]:
print(f"Small embedding length: {min(len(e) for e in embeddings)}")

Small embedding length: 1024


# Using API

In [47]:
load_dotenv(".env")  # or load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_ENDPOINT"],
    api_key=os.environ["AZURE_API_KEY"],
    api_version=os.getenv("AZURE_API_VERSION", "2024-02-01"),
)

# In Azure, this is your DEPLOYMENT name (not raw model name)
DEPLOY_MEDIUM = os.environ["DEPLOY_MEDIUM"]  # deployed as text-embedding-3-medium
DEPLOY_LARGE = os.environ["DEPLOY_LARGE"]  # deployed as text-embedding-3-large

In [ ]:
response = client.embeddings.create(
    input=res[0],
    model=DEPLOY_MEDIUM
)
print(f"Medium embedding length: {len(response.data[0].embedding)}")

response = client.embeddings.create(
    input=res[0],
    model=DEPLOY_LARGE
)
print(f"Large embedding length: {len(response.data[0].embedding)}")


Medium embedding length: 1536
Large embedding length: 3072


# Make sure you have the database up and running

In [11]:
conn_string = "postgresql://{user}:{password}@{host}:{port}/{database}".format(
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST"),
    port=5431,
    database=os.getenv("PGDATABASE")
)

In [12]:
try:
    conn = psycopg2.connect(conn_string)
    print("Connection successful")
    with conn.cursor() as cur:
        register_vector(cur)
        cur.execute("SELECT 1")
        print(cur.fetchone())
except Exception as e:
    print("Connection failed")
    print(e)


Connection failed
could not translate host name "postgresql_database" to address: Name or service not known

